In [1]:
# =========================
# 1. 环境配置：Germany Power BI
# =========================

!pip install PyGithub pandas openpyxl boto3 -q

import pandas as pd
import boto3

from github import Github, Auth
from datetime import datetime
from getpass import getpass


# -------------------------
# AWS 配置
# 输入时不会显示，也不会保存进 Notebook
# -------------------------
AWS_ACCESS_KEY_ID = "YOUR_AWS_ACCESS_KEY_ID"
AWS_SECRET_ACCESS_KEY = "YOUR_AWS_SECRET_ACCESS_KEY"

REGION_NAME = "eu-north-1"

# bucket
BUCKET_NAME = "mds7-hui-zhang-titanic"


# -------------------------
# GitHub 配置
# 输入 GitHub Personal Access Token；不会显示或保存
# -------------------------
STUDENT_TOKEN = "YOUR_GITHUB_TOKEN"

# 改为你自己的 GitHub 用户名/仓库名
REPO_NAME = "ZhangHui88888/mds7-Hui-Zhang"

TARGET_FOLDER = "week-05-06-bigquery/PowerBI"

# 文件名
excel_filename = "Germany-Power-Bi-Dataset.xlsx"
csv_filename = "germany_sales_clean.csv"
pdf_filename = "Taskmds7_week5.pdf"


# -------------------------
# 创建客户端
# -------------------------
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME
)

auth = Auth.Token(STUDENT_TOKEN)
g = Github(auth=auth)
repo = g.get_repo(REPO_NAME)

print("Environment setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.7 MB/s eta 0:00:00
Environment setup complete.


In [10]:
# =========================
# 2. 清洗 Germany 数据
# =========================

from google.colab import files

uploaded = files.upload()

df = pd.read_excel(
    excel_filename,
    sheet_name="Data"
)

# 标准化日期
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

# 用 Date 重新生成正确的英文月份
df["Month"] = df["Date"].dt.month_name()

# 计算总收入
df["Total Revenue"] = (
    df["Price per Unit"] * df["Units Sold"]
)

# 检查日期转换
print("Invalid dates:", df["Date"].isna().sum())

# 保存标准 CSV
df.to_csv(
    csv_filename,
    index=False,
    date_format="%Y-%m-%d"
)

print(f"Clean CSV recreated: {csv_filename}")
df[["Date", "Month", "Total Revenue"]].head()

Saving Germany-Power-Bi-Dataset.xlsx to Germany-Power-Bi-Dataset (2).xlsx
Invalid dates: 0
Clean CSV recreated: germany_sales_clean.csv


,Date,Month,Total Revenue
0,2022-01-14,January,5520.0
1,2022-01-14,January,4600.0
2,2022-01-14,January,3700.0
3,2022-01-14,January,3485.0
4,2022-01-14,January,4950.0


In [11]:
# =========================
# 3. 上传 clean CSV 到 S3 和 GitHub
# =========================

from github import GithubException

# 使用仓库实际的默认分支
TARGET_BRANCH = repo.default_branch

github_path = f"{TARGET_FOLDER}/{csv_filename}"
s3_key = f"{TARGET_FOLDER}/{csv_filename}"


# -------------------------
# 3.1 上传到 AWS S3
# -------------------------
s3_client.upload_file(
    csv_filename,
    BUCKET_NAME,
    s3_key
)

print(f"Uploaded to S3:")
print(f"s3://{BUCKET_NAME}/{s3_key}")


# -------------------------
# 3.2 读取 CSV 内容
# -------------------------
with open(csv_filename, "r", encoding="utf-8") as file:
    csv_content = file.read()


# -------------------------
# 3.3 上传或更新 GitHub
# -------------------------
try:
    existing_file = repo.get_contents(
        github_path,
        ref=TARGET_BRANCH
    )

    repo.update_file(
        path=github_path,
        message="Update clean Germany sales data",
        content=csv_content,
        sha=existing_file.sha,
        branch=TARGET_BRANCH
    )

    github_action = "Updated"

except GithubException as error:
    if error.status == 404:
        repo.create_file(
            path=github_path,
            message="Add clean Germany sales data",
            content=csv_content,
            branch=TARGET_BRANCH
        )

        github_action = "Created"
    else:
        raise


print(f"\n{github_action} on GitHub:")
print(f"Branch: {TARGET_BRANCH}")
print(f"Path: {github_path}")


# -------------------------
# 3.4 输出 Power BI Raw URL
# -------------------------
raw_csv_url = (
    f"https://raw.githubusercontent.com/"
    f"{REPO_NAME}/{TARGET_BRANCH}/{github_path}"
)

print("\nPower BI Raw CSV URL:")
print(raw_csv_url)

Uploaded to S3:
s3://mds7-hui-zhang-titanic/week-05-06-bigquery/PowerBI/germany_sales_clean.csv

Updated on GitHub:
Branch: main
Path: week-05-06-bigquery/PowerBI/germany_sales_clean.csv

Power BI Raw CSV URL:
https://raw.githubusercontent.com/ZhangHui88888/mds7-Hui-Zhang/main/week-05-06-bigquery/PowerBI/germany_sales_clean.csv


In [9]:
# =========================
# 4. 注入极端数据，演示 Power BI 实时刷新
# =========================

import pandas as pd

# 保存干净数据副本，方便演示后恢复
df_clean = df.copy()

# 创建 5 行极端销量数据
demo_rows = pd.DataFrame({
    "Retailer": ["Rewe"] * 5,
    "Retailer ID": [1185732] * 5,
    "Date": [pd.Timestamp("2022-12-31")] * 5,
    "Month": ["December"] * 5,
    "Region": ["Northeast"] * 5,
    "State": ["Berlin"] * 5,
    "Beverage Brand": ["Fritz-Kola"] * 5,
    "Price per Unit": [2.50] * 5,
    "Units Sold": [500000] * 5,
    "Total Revenue": [1250000] * 5
})

# 将演示数据追加到原始数据
df_demo = pd.concat(
    [df_clean, demo_rows],
    ignore_index=True
)

# 覆盖生成演示版 CSV
df_demo.to_csv(
    csv_filename,
    index=False,
    date_format="%Y-%m-%d"
)

print("Demo rows added:", len(demo_rows))
print("New total rows:", len(df_demo))


# 更新 GitHub 中的 CSV
with open(csv_filename, "r", encoding="utf-8") as file:
    demo_content = file.read()

existing_file = repo.get_contents(
    github_path,
    ref=TARGET_BRANCH
)

repo.update_file(
    path=github_path,
    message="Demo live refresh with extreme Germany sales data",
    content=demo_content,
    sha=existing_file.sha,
    branch=TARGET_BRANCH
)

print("Demo data updated on GitHub.")
print(raw_csv_url)

Demo rows added: 5
New total rows: 3744
Demo data updated on GitHub.
https://raw.githubusercontent.com/ZhangHui88888/mds7-Hui-Zhang/main/week-05-06-bigquery/PowerBI/germany_sales_clean.csv
